In [ ]:
#add libraries for tabular and vector data and for plotting
import pandas as pd #tab
import geopandas as gpd #vector
import matplotlib.pyplot as plt #basic plotting
import seaborn as sns #advanced plotting
#mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# read tab separated file
file = '/content/drive/My Drive/ESIIL2025MSU/Project Data/cnty_water_use'
#define the variable
water_use = pd.read_csv(file, sep='\t', comment='#', skiprows=11) # skip meta data rows
display(water_use.head())
print(water_use.columns)

,state_cd,state_name,county_cd,county_nm,year,"Total Population total population of area, in thousands","Public Supply population served by groundwater, in thousands","Public Supply population served by surface water, in thousands","Public Supply total population served, in thousands","Public Supply self-supplied groundwater withdrawals, fresh, in Mgal/d",...,"Hydroelectric Power power generated by instream use, in gigawatt-hours","Hydroelectric Power power generated by offstream use, in gigawatt-hours","Hydroelectric Power total power generated, in gigawatt-hours",Hydroelectric Power number of instream facilities,Hydroelectric Power number of offstream facilities,Hydroelectric Power total number of facilities,"Wastewater Treatment returns by public wastewater facilities, in Mgal/d",Wastewater Treatment number of public wastewater facilities,Wastewater Treatment number of wastewater facilities,"Wastewater Treatment reclaimed wastewater released by wastewater facilities, in Mgal/d"
0,8,Colorado,1,Adams County,1985,276.470,65.650,202.250,267.900,11.18,...,0.00,-,-,-,-,-,162.36,11,-,-
1,8,Colorado,1,Adams County,1990,265.040,49.390,205.380,254.770,9.03,...,0.00,-,-,0,-,-,168.60,9,-,-
2,8,Colorado,1,Adams County,1995,303.300,58.820,230.400,289.220,9.61,...,0.00,0.00,0.00,0,0,0,165.72,14,-,-
3,8,Colorado,1,Adams County,2000,363.860,-,-,328.780,4.30,...,-,-,-,-,-,-,-,-,-,-
4,8,Colorado,1,Adams County,2005,399.426,81.990,301.458,383.448,12.24,...,-,-,-,-,-,-,-,-,-,-


Index(['state_cd', 'state_name', 'county_cd', 'county_nm', 'year',
       'Total Population total population of area, in thousands',
       'Public Supply population served by groundwater, in thousands',
       'Public Supply population served by surface water, in thousands',
       'Public Supply total population served, in thousands',
       'Public Supply self-supplied groundwater withdrawals, fresh, in Mgal/d',
       ...
       'Hydroelectric Power power generated by instream use, in gigawatt-hours',
       'Hydroelectric Power power generated by offstream use, in gigawatt-hours',
       'Hydroelectric Power total power generated, in gigawatt-hours',
       'Hydroelectric Power number of instream facilities',
       'Hydroelectric Power number of offstream facilities',
       'Hydroelectric Power total number of facilities',
       'Wastewater Treatment returns by public wastewater facilities, in Mgal/d',
       'Wastewater Treatment number of public wastewater facilities',
      

In [ ]:
# Create a list of columns with the substring 'Groundwater'
cols_groundwater = [col for col in water_use.columns if 'groundwater' in col]
print(cols_groundwater)
#print length of cols_groundwater
print(len(cols_groundwater))

['Public Supply population served by groundwater, in thousands', 'Public Supply self-supplied groundwater withdrawals, fresh, in Mgal/d', 'Public Supply self-supplied groundwater withdrawals, saline, in Mgal/d', 'Public Supply total self-supplied withdrawals, groundwater, in Mgal/d', 'Domestic self-supplied groundwater withdrawals, fresh, in Mgal/d', 'Domestic self-supplied groundwater withdrawals, saline, in Mgal/d', 'Domestic total self-supplied withdrawals, groundwater, in Mgal/d', 'Commercial self-supplied groundwater withdrawals, fresh, in Mgal/d', 'Commercial self-supplied groundwater withdrawals, saline, in Mgal/d', 'Commercial total self-supplied withdrawals, groundwater, in Mgal/d', 'Industrial self-supplied groundwater withdrawals, fresh, in Mgal/d', 'Industrial self-supplied groundwater withdrawals, saline, in Mgal/d', 'Industrial total self-supplied withdrawals, groundwater, in Mgal/d', 'Total Thermoelectric Power self-supplied groundwater withdrawals, fresh, in Mgal/d', 'T

In [ ]:
# create a new dataframe with only the columns in cols_groundwater and 'year'
ground_wateruse = water_use[['year', 'county_nm', 'Total Population total population of area, in thousands'] + cols_groundwater].copy()

#keep only necessary columns
ground_wateruse=ground_wateruse[['year', 'county_nm',
'Total Population total population of area, in thousands',
'Public Supply self-supplied groundwater withdrawals, fresh, in Mgal/d',
'Domestic self-supplied groundwater withdrawals, fresh, in Mgal/d',
'Industrial self-supplied groundwater withdrawals, fresh, in Mgal/d',
'Total Thermoelectric Power self-supplied groundwater withdrawals, fresh, in Mgal/d',
'Mining total self-supplied withdrawals, groundwater, in Mgal/d',
'Livestock self-supplied groundwater withdrawals, fresh, in Mgal/d',
'Livestock (Stock) self-supplied groundwater withdrawals, fresh, in Mgal/d',
'Irrigation, Total self-supplied groundwater withdrawals, fresh, in Mgal/d',
                   ]]

# convert livestock columns to numeric
ground_wateruse['Livestock self-supplied groundwater withdrawals, fresh, in Mgal/d'] = pd.to_numeric(ground_wateruse['Livestock self-supplied groundwater withdrawals, fresh, in Mgal/d'], errors='coerce').fillna(0)
ground_wateruse['Livestock (Stock) self-supplied groundwater withdrawals, fresh, in Mgal/d'] = pd.to_numeric(ground_wateruse['Livestock (Stock) self-supplied groundwater withdrawals, fresh, in Mgal/d'], errors='coerce').fillna(0)
#calculate the sum of agriculture columns
ground_wateruse['Agriculture_Self'] = ground_wateruse['Irrigation, Total self-supplied groundwater withdrawals, fresh, in Mgal/d'] + ground_wateruse['Livestock self-supplied groundwater withdrawals, fresh, in Mgal/d'] + ground_wateruse['Livestock (Stock) self-supplied groundwater withdrawals, fresh, in Mgal/d']
#drop agriculture columns
ground_wateruse.drop(columns=['Livestock self-supplied groundwater withdrawals, fresh, in Mgal/d', 'Livestock (Stock) self-supplied groundwater withdrawals, fresh, in Mgal/d','Irrigation, Total self-supplied groundwater withdrawals, fresh, in Mgal/d'], inplace=True)

#Rename necessary columns
ground_wateruse.rename(columns={'Total Population total population of area, in thousands': 'PopTotalK',
  'Public Supply self-supplied groundwater withdrawals, fresh, in Mgal/d': 'Public_Supply',
  'Domestic self-supplied groundwater withdrawals, fresh, in Mgal/d': 'Domestic_Self',
  'Industrial self-supplied groundwater withdrawals, fresh, in Mgal/d': 'Industrial_Self',
  'Total Thermoelectric Power self-supplied groundwater withdrawals, fresh, in Mgal/d': 'Thermoelectric_Self',
  'Mining total self-supplied withdrawals, groundwater, in Mgal/d': 'Mining_Self',
  }, inplace=True)

# Calculate the sum of all columns in groundwateruse, excluding 'year' and 'county_nm'
ground_wateruse['Ground_Use'] = ground_wateruse.drop(columns=['year', 'county_nm', 'PopTotalK']).sum(axis=1)

# Display the updated dataframe
print(ground_wateruse.columns)
print(ground_wateruse.dtypes)
display(ground_wateruse.head())

Index(['year', 'county_nm', 'PopTotalK', 'Public_Supply', 'Domestic_Self',
       'Industrial_Self', 'Thermoelectric_Self', 'Mining_Self',
       'Agriculture_Self', 'Ground_Use'],
      dtype='object')
year                     int64
county_nm               object
PopTotalK              float64
Public_Supply          float64
Domestic_Self          float64
Industrial_Self        float64
Thermoelectric_Self    float64
Mining_Self            float64
Agriculture_Self       float64
Ground_Use             float64
dtype: object


,year,county_nm,PopTotalK,Public_Supply,Domestic_Self,Industrial_Self,Thermoelectric_Self,Mining_Self,Agriculture_Self,Ground_Use
0,1985,Adams County,276.470,11.18,0.64,0.54,0.00,1.90,37.23,51.49
1,1990,Adams County,265.040,9.03,0.78,3.32,0.00,1.75,46.15,61.03
2,1995,Adams County,303.300,9.61,1.06,3.52,0.00,2.68,37.36,54.23
3,2000,Adams County,363.860,4.30,4.21,3.50,0.00,2.82,15.28,30.11
4,2005,Adams County,399.426,12.24,1.37,0.71,0.01,0.17,1.19,15.69


In [ ]:
# Create a list of columns with the substring 'surface'
cols_surfacewater = [col for col in water_use.columns if 'surface' in col]
print(cols_surfacewater)
#print length of cols_surfacewater
print(len(cols_surfacewater))

['Public Supply population served by surface water, in thousands', 'Public Supply self-supplied surface-water withdrawals, fresh, in Mgal/d', 'Public Supply self-supplied surface-water withdrawals, saline, in Mgal/d', 'Public Supply total self-supplied withdrawals, surface water, in Mgal/d', 'Domestic self-supplied surface-water withdrawals, fresh, in Mgal/d', 'Domestic self-supplied surface-water withdrawals, saline, in Mgal/d', 'Domestic total self-supplied withdrawals, surface water, in Mgal/d', 'Commercial self-supplied surface-water withdrawals, fresh, in Mgal/d', 'Commercial self-supplied surface-water withdrawals, saline, in Mgal/d', 'Commercial total self-supplied withdrawals, surface water, in Mgal/d', 'Industrial self-supplied surface-water withdrawals, fresh, in Mgal/d', 'Industrial self-supplied surface-water withdrawals, saline, in Mgal/d', 'Industrial total self-supplied withdrawals, surface water, in Mgal/d', 'Total Thermoelectric Power self-supplied surface-water withdr

In [ ]:
# create a new dataframe with only the columns in cols_surfacewater, 'year', and 'county_nm'
surface_wateruse = water_use[['year', 'county_nm', 'Total Population total population of area, in thousands'] + cols_surfacewater].copy()
#keep only necessary columns
surface_wateruse=surface_wateruse[['year', 'county_nm',
'Total Population total population of area, in thousands',
'Public Supply self-supplied surface-water withdrawals, fresh, in Mgal/d',
'Domestic self-supplied surface-water withdrawals, fresh, in Mgal/d',
'Industrial self-supplied surface-water withdrawals, fresh, in Mgal/d',
'Total Thermoelectric Power self-supplied surface-water withdrawals, fresh, in Mgal/d',
'Fossil-fuel Thermoelectric Power total self-supplied withdrawals, surface water, in Mgal/d',
'Nuclear Thermoelectric Power total self-supplied withdrawals, surface water, in Mgal/d',
'Mining total self-supplied withdrawals, surface water, in Mgal/d',
'Livestock self-supplied surface-water withdrawals, fresh, in Mgal/d',
'Livestock (Stock) self-supplied surface-water withdrawals, fresh, in Mgal/d',
'Irrigation, Total self-supplied surface-water withdrawals, fresh, in Mgal/d',
                   ]].copy()

# combine the thermoelectric columns
#convert electric to numeric
surface_wateruse['Total Thermoelectric Power self-supplied surface-water withdrawals, fresh, in Mgal/d'] = pd.to_numeric(surface_wateruse['Total Thermoelectric Power self-supplied surface-water withdrawals, fresh, in Mgal/d'], errors='coerce')
surface_wateruse['Nuclear Thermoelectric Power total self-supplied withdrawals, surface water, in Mgal/d'] = pd.to_numeric(surface_wateruse['Nuclear Thermoelectric Power total self-supplied withdrawals, surface water, in Mgal/d'], errors='coerce')
surface_wateruse['Fossil-fuel Thermoelectric Power total self-supplied withdrawals, surface water, in Mgal/d'] = pd.to_numeric(surface_wateruse['Fossil-fuel Thermoelectric Power total self-supplied withdrawals, surface water, in Mgal/d'], errors='coerce')
# create a new column for Thermoelectric_Self by combining the relevant columns,
surface_wateruse['Thermoelectric_Self'] = surface_wateruse['Fossil-fuel Thermoelectric Power total self-supplied withdrawals, surface water, in Mgal/d'].fillna(0) + surface_wateruse['Nuclear Thermoelectric Power total self-supplied withdrawals, surface water, in Mgal/d'].fillna(0)
# surface_wateruse['Thermoelectric_Self] = max of surface_wateruse['Thermoelectric_Self] and surface_wateruse['Total Thermoelectric Power self-supplied surface-water withdrawals, fresh, in Mgal/d']
surface_wateruse['Thermoelectric_Self'] = surface_wateruse[['Thermoelectric_Self', 'Total Thermoelectric Power self-supplied surface-water withdrawals, fresh, in Mgal/d']].max(axis=1)
#drop the unecessary columns
surface_wateruse  = surface_wateruse.drop(columns=['Total Thermoelectric Power self-supplied surface-water withdrawals, fresh, in Mgal/d', 'Nuclear Thermoelectric Power total self-supplied withdrawals, surface water, in Mgal/d', 'Fossil-fuel Thermoelectric Power total self-supplied withdrawals, surface water, in Mgal/d'])


# combine the livestock columns
# convert livestock columns to numeric
surface_wateruse['Livestock self-supplied surface-water withdrawals, fresh, in Mgal/d'] = pd.to_numeric(surface_wateruse['Livestock self-supplied surface-water withdrawals, fresh, in Mgal/d'], errors='coerce')
surface_wateruse['Livestock (Stock) self-supplied surface-water withdrawals, fresh, in Mgal/d'] = pd.to_numeric(surface_wateruse['Livestock (Stock) self-supplied surface-water withdrawals, fresh, in Mgal/d'], errors='coerce')
#calculate the sum of livestock columns
surface_wateruse['Agriculture_Self'] = surface_wateruse['Irrigation, Total self-supplied surface-water withdrawals, fresh, in Mgal/d'] + surface_wateruse['Livestock self-supplied surface-water withdrawals, fresh, in Mgal/d'].fillna(0) + surface_wateruse['Livestock (Stock) self-supplied surface-water withdrawals, fresh, in Mgal/d'].fillna(0)
#drop livestock columns
surface_wateruse.drop(columns=['Irrigation, Total self-supplied surface-water withdrawals, fresh, in Mgal/d','Livestock self-supplied surface-water withdrawals, fresh, in Mgal/d', 'Livestock (Stock) self-supplied surface-water withdrawals, fresh, in Mgal/d'], inplace=True)

#Rename necessary columns
surface_wateruse.rename(columns={'Total Population total population of area, in thousands': 'PopTotalK',
  'Public Supply self-supplied surface-water withdrawals, fresh, in Mgal/d': 'Public_Supply',
  'Domestic self-supplied surface-water withdrawals, fresh, in Mgal/d': 'Domestic_Self',
  'Industrial self-supplied surface-water withdrawals, fresh, in Mgal/d': 'Industrial_Self',
  'Mining total self-supplied withdrawals, surface water, in Mgal/d': 'Mining_Self',
  }, inplace=True)

# Calculate the sum of all columns in surface_wateruse, excluding 'year'
surface_wateruse['Surface_Use'] = surface_wateruse.drop(columns=['year', 'county_nm']).sum(axis=1)
# Display the updated dataframe
print(surface_wateruse.columns)
print(surface_wateruse.dtypes)
display(surface_wateruse.head())

Index(['year', 'county_nm', 'PopTotalK', 'Public_Supply', 'Domestic_Self',
       'Industrial_Self', 'Mining_Self', 'Thermoelectric_Self',
       'Agriculture_Self', 'Surface_Use'],
      dtype='object')
year                     int64
county_nm               object
PopTotalK              float64
Public_Supply          float64
Domestic_Self          float64
Industrial_Self        float64
Mining_Self            float64
Thermoelectric_Self    float64
Agriculture_Self       float64
Surface_Use            float64
dtype: object


,year,county_nm,PopTotalK,Public_Supply,Domestic_Self,Industrial_Self,Mining_Self,Thermoelectric_Self,Agriculture_Self,Surface_Use
0,1985,Adams County,276.470,43.57,0.0,0.06,0.0,0.00,32.80,352.900
1,1990,Adams County,265.040,37.82,0.0,1.18,0.0,0.00,37.79,341.830
2,1995,Adams County,303.300,37.94,0.0,1.26,0.1,0.00,38.58,381.180
3,2000,Adams County,363.860,70.75,0.0,2.97,0.1,3.16,29.74,470.580
4,2005,Adams County,399.426,38.83,0.0,1.73,0.0,9.24,111.77,560.996


In [ ]:
# Sum the corresponding columns of the two dataframes
total_wateruse = surface_wateruse.copy()

#ensure columns match
cols_to_sum = [col for col in surface_wateruse.columns if col not in ['year', 'county_nm', 'PopTotalK']]

for col in cols_to_sum:
  # Ensure columns exist in both dataframes before summing
  if col in ground_wateruse.columns:
    total_wateruse[col] = surface_wateruse[col] + ground_wateruse[col]
  else:
      # If a column exists only in surface_wateruse, keep its value
      total_wateruse[col] = surface_wateruse[col]

# Now handle columns that might exist only in ground_wateruse
# This loop is for robustness in case the attributes aren't perfectly identical.
for col in ground_wateruse.columns:
    if col not in ['year', 'county_nm'] and col not in total_wateruse.columns:
        total_wateruse[col] = ground_wateruse[col]

total_wateruse['Total_Use'] = total_wateruse['Surface_Use'] + total_wateruse['Ground_Use']

# Display the resulting dataframe
display(total_wateruse)

,year,county_nm,PopTotalK,Public_Supply,Domestic_Self,Industrial_Self,Mining_Self,Thermoelectric_Self,Agriculture_Self,Surface_Use,Ground_Use,Total_Use
0,1985,Adams County,276.470,54.75,0.64,0.60,1.90,0.00,70.03,352.900,51.49,404.390
1,1990,Adams County,265.040,46.85,0.78,4.50,1.75,0.00,83.94,341.830,61.03,402.860
2,1995,Adams County,303.300,47.55,1.06,4.78,2.78,0.00,75.94,381.180,54.23,435.410
3,2000,Adams County,363.860,75.05,4.21,6.47,2.92,3.16,45.02,470.580,30.11,500.690
4,2005,Adams County,399.426,51.07,1.37,2.44,0.17,9.25,112.96,560.996,15.69,576.686
...,...,...,...,...,...,...,...,...,...,...,...,...
439,1995,Yuma County,9.260,2.40,0.27,0.00,0.02,0.00,270.03,13.680,268.30,281.980
440,2000,Yuma County,9.840,0.83,0.51,0.00,0.08,0.00,323.16,16.210,318.21,334.420
441,2005,Yuma County,9.789,2.07,0.83,0.00,0.18,0.00,347.76,18.059,342.57,360.629
442,2010,Yuma County,10.043,1.31,0.53,0.00,0.13,0.00,215.12,12.113,215.02,227.133


In [ ]:
# Calculate the Pearson correlation coefficient
corr_Public_Ag = total_wateruse['Public_Supply'].corr(total_wateruse['Agriculture_Self'], method='pearson')

print(f"The Pearson correlation coefficient between Public_Supply and Agriculture_Self is: {corr_Public_Ag}")

The Pearson correlation coefficient between Public_Supply and Agriculture_Self is: -0.0713823562808645


In [ ]:
# Calculate the Pearson correlation coefficient between 'PopTotalK' and 'Agriculture_Self' grouped by year
corr_pop_Ag = total_wateruse.groupby('year')['PopTotalK'].corr(total_wateruse['Agriculture_Self'], method='pearson')

print(f"Pearson correlation coefficient between PopTotalK and Agriculture_Self: {corr_pop_Ag}")

Pearson correlation coefficient between PopTotalK and Agriculture_Self: year
1985   -0.038233
1990   -0.046858
1995   -0.102362
2000   -0.085842
2005   -0.093653
2010   -0.064996
2015   -0.119269
Name: PopTotalK, dtype: float64


In [ ]:
# Calculate the Pearson correlation coefficient between 'PopTotalK' and 'Public_Supply'
correlation_pop_public = total_wateruse['PopTotalK'].corr(total_wateruse['Public_Supply'], method='pearson')

print(f"Pearson correlation coefficient between PopTotalK and Public_Supply: {correlation_pop_public}")

Pearson correlation coefficient between PopTotalK and Public_Supply: 0.9112541875941261


In [ ]:
#create invidual dataframes for the counties
douglas_wateruse = total_wateruse[total_wateruse['county_nm'].isin(['Douglas County'])].copy()
eagle_wateruse = total_wateruse[total_wateruse['county_nm'].isin(['Eagle County'])].copy()
logan_wateruse = total_wateruse[total_wateruse['county_nm'].isin(['Logan County'])].copy()
display(douglas_wateruse)
display(eagle_wateruse)
display(logan_wateruse)

,year,county_nm,PopTotalK,Public_Supply,Domestic_Self,Industrial_Self,Mining_Self,Thermoelectric_Self,Agriculture_Self,Surface_Use,Ground_Use,Total_Use
122,1985,Douglas County,36.760,4.19,1.18,0.95,0.10,0.0,8.09,43.980,7.29,51.270
123,1990,Douglas County,60.390,8.04,1.42,0.54,0.10,0.0,14.37,73.840,11.02,84.860
124,1995,Douglas County,99.580,13.36,2.34,0.09,0.20,0.0,12.59,115.750,12.41,128.160
125,2000,Douglas County,175.770,27.47,4.97,0.00,0.29,0.0,11.06,203.480,16.08,219.560
126,2005,Douglas County,249.416,30.16,0.96,0.01,0.03,0.0,16.86,277.456,19.98,297.436
127,2010,Douglas County,285.465,36.17,0.93,0.00,0.00,0.0,15.81,312.555,25.82,338.375
128,2015,Douglas County,322.387,38.48,1.71,0.00,0.04,0.0,11.90,345.677,28.84,374.517


,year,county_nm,PopTotalK,Public_Supply,Domestic_Self,Industrial_Self,Mining_Self,Thermoelectric_Self,Agriculture_Self,Surface_Use,Ground_Use,Total_Use
129,1985,Eagle County,18.260,6.03,0.04,0.00,0.03,0.0,218.09,239.240,3.21,242.450
130,1990,Eagle County,21.930,6.19,0.17,0.21,0.03,0.0,177.33,201.350,4.51,205.860
131,1995,Eagle County,28.840,8.86,0.21,0.21,0.22,0.0,127.79,161.320,4.81,166.130
132,2000,Eagle County,41.660,10.11,1.02,0.00,0.33,0.0,129.62,179.960,2.78,182.740
133,2005,Eagle County,47.530,9.19,0.01,0.26,0.04,0.0,145.75,198.680,4.10,202.780
134,2010,Eagle County,52.197,10.70,0.76,0.02,0.15,0.0,126.66,185.127,5.36,190.487
135,2015,Eagle County,53.605,10.31,0.40,0.00,0.21,0.0,132.23,191.315,5.44,196.755


,year,county_nm,PopTotalK,Public_Supply,Domestic_Self,Industrial_Self,Mining_Self,Thermoelectric_Self,Agriculture_Self,Surface_Use,Ground_Use,Total_Use
262,1985,Logan County,19.820,3.35,0.50,0.61,1.84,0.0,237.26,182.940,80.44,263.380
263,1990,Logan County,17.570,3.05,0.40,0.00,1.09,0.0,275.70,202.530,95.28,297.810
264,1995,Logan County,17.870,4.81,0.40,1.24,0.50,0.0,328.61,254.450,98.98,353.430
265,2000,Logan County,20.500,2.10,0.78,0.71,0.54,0.0,307.85,274.620,57.86,332.480
266,2005,Logan County,20.719,2.54,0.43,0.00,0.56,0.0,282.94,301.479,5.71,307.189
267,2010,Logan County,22.709,2.98,0.68,0.00,0.31,0.0,160.23,127.929,58.98,186.909
268,2015,Logan County,22.036,3.26,0.68,0.55,0.44,0.0,145.55,106.826,65.69,172.516


In [ ]:
!pip install plotly
import plotly.express as px

def plot_interactive_water_use(df, county_name, color_map=None):
  """
  Creates an interactive line plot of water use categories over the years for a specific county.

  Args:
    df: pandas DataFrame containing water use data.
    county_name: String, the name of the county to plot.
    color_map: Dictionary, mapping column names to colors.
  """
  county_df = df[df['county_nm'] == county_name].copy()

  # Exclude some columns and columns where the value is 0 for all years
  columns_to_plot = [col for col in county_df.columns if col not in ['year',
                                                                     'county_nm',
                                                                     'Total_Use',
                                                                     'Ground_Use',
                                                                     'Surface_Use',
                                                                     'PopTotalK']
                      and not (county_df[col] == 0).all()]
# Melt the DataFrame
  county_melted = county_df.melt(id_vars=['year', 'county_nm'],
                                  value_vars=columns_to_plot,
                                  var_name='Category',
                                  value_name='Water Use (Mgal/d)')

  # Create a mapping for more descriptive labels
  label_map = {
      'Public_Supply': 'Public Supply',
      'Domestic_Self': 'Domestic (self-supplied)',
      'Industrial_Self': 'Industrial (self-supplied)',
      'Thermoelectric_Self': 'Thermoelectric (self-supplied)',
      'Mining_Self': 'Mining (self-supplied)',
      'Agriculture_Self': 'Agriculture (self-supplied)'
  }

  # Apply label transformation
  county_melted['Category'] = county_melted['Category'].replace(label_map)

  # Calculate peak value for each category and sort so that legend labels appear in the same orders as lines
  peak_values = county_melted.groupby('Category')['Water Use (Mgal/d)'].max().sort_values(ascending=False)
  sorted_categories = peak_values.index.tolist()

  # Create a color map with the descriptive labels as keys
  descriptive_color_map = {label_map[col]: color_map[col] for col in columns_to_plot if col in label_map}


  fig = px.line(county_melted, x='year', y='Water Use (Mgal/d)', color='Category',
                title=f'Water Withdrawal Use Over Time: {county_name}',
                labels={'year': 'Year', 'Water Use (Mgal/d)': 'Water Use (Mgal/d)', 'Category': 'Category'},
                hover_name='county_nm',
                category_orders={'Category': sorted_categories}, # Set the order of categories
                color_discrete_map=descriptive_color_map) # Use the provided color_map

  # Update layout for increased font sizes
  fig.update_layout(
      title_font_size=20,  # Increase title font size
      legend_title_font_size=16, # Increase legend title font size
      legend_font_size=14, # Increase legend label font size
      xaxis=dict(
          title_font_size=16,  # Increase x-axis label font size
          tickfont_size=14  # Increase x-axis tick font size
      ),
      yaxis=dict(
          title_font_size=14,  # Increase y-axis label font size
          tickfont_size=14  # Increase y-axis tick font size
      )
  )

  fig.show()

  # Save the plot as an HTML file
  file_name = f'water_uses_{county_name.replace(" ", "_")}.html'
  fig.write_html(file_name)
  print(f"Saved plot for {county_name} as {file_name}")



# Define a consistent color map for all plots
# You can customize these colors as needed
color_map = {
    'Public_Supply': 'blue',
    'Domestic_Self': 'red',
    'Industrial_Self': 'green',
    'Thermoelectric_Self': 'purple',
    'Mining_Self': 'orange',
    'Agriculture_Self': 'brown'
}


# Example usage:
plot_interactive_water_use(total_wateruse, 'Douglas County', color_map)
plot_interactive_water_use(total_wateruse, 'Eagle County', color_map)
plot_interactive_water_use(total_wateruse, 'Logan County', color_map)


Saved plot for Douglas County as water_uses_Douglas_County.html


Saved plot for Eagle County as water_uses_Eagle_County.html


Saved plot for Logan County as water_uses_Logan_County.html


In [ ]:
def plot_stacked_water_use(df, county_name, color_map=None):
  """
  Creates an interactive stacked area chart of water use categories over the years for a specific county.

  Args:
    df: pandas DataFrame containing water use data.
    county_name: String, the name of the county to plot.
    color_map: Dictionary, mapping column names to colors.
  """
  county_df = df[df['county_nm'] == county_name].copy()

   # Exclude some columns and columns where the value is 0 for all years
  columns_to_plot = [col for col in county_df.columns if col not in ['year',
                                                                     'county_nm',
                                                                     'Public_Supply',
                                                                     'Domestic_Self',
                                                                     'Industrial_Self',
                                                                     'Thermoelectric_Self',
                                                                     'Mining_Self',
                                                                     'Agriculture_Self',
                                                                     'Total_Use',
                                                                     'PopTotalK']
                      and not (county_df[col] == 0).all()]

  # Define a consistent color map for all plots
  color_map = {'Surface_Use':'teal',
              'Ground_Use':'brown'
  }

  # Melt the DataFrame
  county_melted = county_df.melt(id_vars=['year', 'county_nm'],
                                  value_vars=columns_to_plot,
                                  var_name='Category',
                                  value_name='Water Use (Mgal/d)')

  # Create a mapping for more descriptive labels
  label_map = {'Surface_Use':'Surfacewater Withdrawals',
               'Ground_Use':'Groundwater Withdrawals',
  }

  # Apply label transformation
  county_melted['Category'] = county_melted['Category'].replace(label_map)

  # Calculate peak value for each category and sort
  peak_values = county_melted.groupby('Category')['Water Use (Mgal/d)'].max().sort_values(ascending=False)
  sorted_categories = peak_values.index.tolist()

  fig = px.line(county_melted, x='year', y='Water Use (Mgal/d)', color='Category',
                title=f'Water Withdrawal Source Over Time: {county_name}',
                labels={'year': 'Year', 'Water Use (Mgal/d)': 'Water Use (Mgal/d)', 'Category': 'Category'},
                hover_name='county_nm',
                category_orders={'Category': sorted_categories}, # Set the order of categories
                color_discrete_map={label_map[k]: v for k, v in color_map.items() if k in label_map})
  # Update layout for increased font sizes
  fig.update_layout(
      hovermode="x unified", # Unify hover for better comparison
      title_font_size=30,  # Increase title font size
      legend_title_font_size=20, # Increase legend title font size
      xaxis=dict(title_font_size=20, tickfont=dict(size=18)), # Increase x-axis label font size
      yaxis=dict(title_font_size=20, tickfont=dict(size=18)),  # Increase y-axis label and tick font size
      legend=dict(font=dict(size=18)) # Increase legend font size
    )

  fig.show()
  # Save the plot as an HTML file
  file_name = f'water_source_{county_name.replace(" ", "_")}.html'
  fig.write_html(file_name)
  print(f"Saved plot for {county_name} as {file_name}")


# Example usage:
plot_stacked_water_use(total_wateruse, 'Douglas County', color_map)
plot_stacked_water_use(total_wateruse, 'Eagle County', color_map)
plot_stacked_water_use(total_wateruse, 'Logan County', color_map)

Saved plot for Douglas County as water_source_Douglas_County.html


Saved plot for Eagle County as water_source_Eagle_County.html


Saved plot for Logan County as water_source_Logan_County.html


In [ ]:
def plot_stacked_water_use(df, county_name, color_map=None):
  """
  Creates an interactive stacked area chart of water use categories over the years for a specific county.

  Args:
    df: pandas DataFrame containing water use data.
    county_name: String, the name of the county to plot.
    color_map: Dictionary, mapping column names to colors.
  """
  county_df = df[df['county_nm'] == county_name].copy()

   # Exclude some columns and columns where the value is 0 for all years
  columns_to_plot = [col for col in county_df.columns if col not in ['year',
                                                                     'county_nm',
                                                                     'Total_Use',
                                                                     'Ground_Use',
                                                                     'Surface_Use',
                                                                     'PopTotalK']
                      and not (county_df[col] == 0).all()]

  # Melt the DataFrame
  county_melted = county_df.melt(id_vars=['year', 'county_nm'],
                                  value_vars=columns_to_plot,
                                  var_name='Category',
                                  value_name='Water Use (Mgal/d)')

  # Create a mapping for more descriptive labels
  label_map = {
      'Public_Supply': 'Public Supply',
      'Domestic_Self': 'Domestic (self-supplied)',
      'Industrial_Self': 'Industrial (self-supplied)',
      'Thermoelectric_Self': 'Thermoelectric (self-supplied)',
      'Mining_Self': 'Mining (self-supplied)',
      'Agriculture_Self': 'Agriculture (self-supplied)'
  }

  # Apply label transformation
  county_melted['Category'] = county_melted['Category'].replace(label_map)

  # Calculate peak value for each category and sort
  peak_values = county_melted.groupby('Category')['Water Use (Mgal/d)'].max().sort_values(ascending=False)
  sorted_categories = peak_values.index.tolist()

  # Define a consistent color map for all plots
  color_map = {
      'Public_Supply': 'blue',
      'Domestic_Self': 'red',
      'Industrial_Self': 'green',
      'Thermoelectric_Self': 'purple',
      'Mining_Self': 'orange',
      'Agriculture_Self': 'brown'
}

  fig = px.line(county_melted, x='year', y='Water Use (Mgal/d)', color='Category',
                title=f'Stacked Water Use Trends for {county_name}',
                labels={'year': 'Year', 'Water Use (Mgal/d)': 'Water Use (Mgal/d)', 'Category': 'Category'},
                hover_name='county_nm',
                category_orders={'Category': sorted_categories}, # Set the order of categories
                color_discrete_map={label_map[k]: v for k, v in color_map.items() if k in label_map})
  fig.update_layout(
      hovermode="x unified", # Unify hover for better comparison
      title_font_size=30,  # Increase title font size
      legend_title_font_size=20, # Increase legend title font size
      xaxis=dict(title_font_size=20, tickfont=dict(size=18)), # Increase x-axis label font size
      yaxis=dict(title_font_size=20, tickfont=dict(size=18)),  # Increase y-axis label and tick font size
      legend=dict(font=dict(size=18)) # Increase legend font size
  )

  fig.show()
  # Save the plot as an HTML file
  file_name = f'water_uses_{county_name.replace(" ", "_")}.html'
  fig.write_html(file_name)
  print(f"Saved plot for {county_name} as {file_name}")


# Example usage:
plot_stacked_water_use(total_wateruse, 'Douglas County', color_map)
plot_stacked_water_use(total_wateruse, 'Eagle County', color_map)
plot_stacked_water_use(total_wateruse, 'Logan County', color_map)

Saved plot for Douglas County as water_uses_Douglas_County.html


Saved plot for Eagle County as water_uses_Eagle_County.html


Saved plot for Logan County as water_uses_Logan_County.html
